In [32]:
%pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [33]:
%pip install langchain langchain-community langchain-text-splitters transformers torch scikit-learn pillow faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [34]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [35]:
%pip install langchain langchain-core langchain-community langchain-text-splitters transformers torch scikit-learn pillow faiss-cpu pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [36]:
pip install -U python-dotenv langchain-google-genai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [37]:
###Clip Model
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("Gemini API key not found!")

print("Gemini API key loaded successfully!")


### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Gemini API key loaded successfully!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [38]:
### Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data
    
    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()
    
def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77  # CLIP's max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


In [39]:
## Process PDF

pdf_path = "sample.pdf"
doc = fitz.open(pdf_path)

# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

print("PDF loaded successfully!")
print("Total pages:", len(doc))

PDF loaded successfully!
Total pages: 1


In [40]:
doc

Document('sample.pdf')

In [41]:
pdf_path = "sample.pdf"
doc = fitz.open(pdf_path)

print("PDF loaded successfully!")
print("Total pages:", len(doc))

PDF loaded successfully!
Total pages: 1


In [42]:
all_docs = []
all_embeddings = []
image_data_store = {}

In [43]:

## MULTIMODAL PDF PROCESSING

import os
import fitz
import io
import base64
import torch

from PIL import Image
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Open PDF
pdf_path = "sample.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

if os.path.getsize(pdf_path) == 0:
    raise ValueError("PDF file is empty!")

doc = fitz.open(pdf_path)

# 2. Initialize storage
all_docs = []
all_embeddings = []
image_data_store = {}

# 3. Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# 4. Generate text embeddings using CLIP
def embed_text(text):

    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = clip_model.text_model(**inputs)
        features = clip_model.text_projection(
            outputs.pooler_output
        )

    return features.squeeze(0).cpu().numpy()


# 5. Generate image embeddings using CLIP
def embed_image(image):

    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        features = clip_model.visual_projection(
            outputs.pooler_output
        )

    return features.squeeze(0).cpu().numpy()


# 6. Process PDF pages
try:

    for i, page in enumerate(doc):

        print(f"Processing page {i + 1}...")

        # PROCESS TEXT
        text = page.get_text()

        if text.strip():

            temp_doc = Document(
                page_content=text,
                metadata={
                    "page": i,
                    "type": "text"
                }
            )

            text_chunks = splitter.split_documents(
                [temp_doc]
            )

            for chunk in text_chunks:

                embedding = embed_text(
                    chunk.page_content
                )

                all_embeddings.append(embedding)
                all_docs.append(chunk)

        # PROCESS IMAGES
        for img_index, img in enumerate(
            page.get_images(full=True)
        ):

            try:

                xref = img[0]

                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                # Convert to PIL Image
                pil_image = Image.open(
                    io.BytesIO(image_bytes)
                ).convert("RGB")

                # Create unique image ID
                image_id = f"page_{i}_img_{img_index}"

                # Convert image to Base64
                buffered = io.BytesIO()

                pil_image.save(
                    buffered,
                    format="PNG"
                )

                img_base64 = base64.b64encode(
                    buffered.getvalue()
                ).decode()

                # Generate image embedding
                embedding = embed_image(pil_image)

                # Create image document
                image_doc = Document(
                    page_content=f"[Image: {image_id}]",
                    metadata={
                        "page": i,
                        "type": "image",
                        "image_id": image_id
                    }
                )

                # Store only after successful processing
                all_embeddings.append(embedding)
                all_docs.append(image_doc)
                image_data_store[image_id] = img_base64

            except Exception as e:

                print(
                    f"Error processing image {img_index} "
                    f"on page {i}: {e}"
                )

                continue

finally:
    doc.close()


# 7. Display processing results
print("\nPDF processing completed!")

print("Total documents:", len(all_docs))
print("Total embeddings:", len(all_embeddings))
print("Total images:", len(image_data_store))

print(
    "Text chunks:",
    sum(
        1 for d in all_docs
        if d.metadata["type"] == "text"
    )
)

print(
    "Image documents:",
    sum(
        1 for d in all_docs
        if d.metadata["type"] == "image"
    )
)

# 8. Verify embedding dimensions
if all_embeddings:
    print(
        "Embedding dimension:",
        all_embeddings[0].shape
    )

    assert len(all_docs) == len(all_embeddings), \
        "Document and embedding counts do not match!"


Processing page 1...

PDF processing completed!
Total documents: 2
Total embeddings: 2
Total images: 1
Text chunks: 1
Image documents: 1
Embedding dimension: (512,)


In [44]:
!pip install -U faiss-cpu langchain-community


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\S Gopi\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [45]:

import os
import numpy as np

from langchain_community.vectorstores import FAISS

# Verify PDF
pdf_path = "sample.pdf"
print("File exists:", os.path.exists(pdf_path))

# Convert precomputed embeddings to NumPy array
embeddings_array = np.array(
    all_embeddings,
    dtype=np.float32
)

# Create FAISS vector database
vector_store = FAISS.from_embeddings(
    text_embeddings=[
        (doc.page_content, emb.tolist())
        for doc, emb in zip(all_docs, embeddings_array)
    ],
    embedding=None,
    metadatas=[
        doc.metadata for doc in all_docs
    ]
)

print("FAISS vector database created successfully!")
print("Total vectors:", vector_store.index.ntotal)

vector_store


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


File exists: True
FAISS vector database created successfully!
Total vectors: 2


In [46]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [47]:

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=api_key
)

print(llm.invoke("What is Multimodal RAG?").content)

[{'type': 'text', 'text': '**Multimodal Retrieval-Augmented Generation (Multimodal RAG)** is an advanced AI framework that extends traditional Retrieval-Augmented Generation (RAG) by supporting multiple types of data modalities—such as **text, images, audio, video, and charts**—instead of just plain text.\n\nLike traditional RAG, it is designed to ground Large Language Models (LLMs) in external knowledge to reduce hallucinations and provide accurate, context-aware answers. However, it can understand and retrieve information from visual and auditory sources, as well as text.\n\n---\n\n### How Multimodal RAG Works\n\nA typical Multimodal RAG pipeline follows these steps:\n\n1. **Ingestion & Processing:** \n   * Documents containing mixed media (e.g., a PDF with text, tables, and graphs) are ingested.\n   * **Text** is chunked normally.\n   * **Images, charts, and diagrams** are processed using Vision-Language Models (VLMs) like GPT-4o, Claude 3.5 Sonnet, or specialized OCR tools to extra

In [48]:
from langchain_core.messages import HumanMessage

In [49]:

from langchain_core.messages import HumanMessage

def create_multimodal_message(query, retrieved_docs):
    """Create a multimodal message for Gemini with text and images."""

    content = []

    # Add user question
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    # Separate text and image documents
    text_docs = [
        doc for doc in retrieved_docs
        if doc.metadata.get("type") == "text"
    ]

    image_docs = [
        doc for doc in retrieved_docs
        if doc.metadata.get("type") == "image"
    ]

    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page'] + 1}]: {doc.page_content}"
            for doc in text_docs
        ])

        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })

    # Add retrieved images
    for doc in image_docs:

        image_id = doc.metadata.get("image_id")

        if image_id and image_id in image_data_store:

            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page'] + 1}]:\n"
            })

            content.append({
                "type": "image_url",
                "image_url": f"data:image/png;base64,{image_data_store[image_id]}"
            })

    # Add final instruction
    content.append({
        "type": "text",
        "text": (
            "\nPlease answer the question based on the "
            "provided text and images. If the context "
            "does not contain the answer, say so."
        )
    })

    return HumanMessage(content=content)


In [50]:
def retrieve_multimodal(query, k=5):
    query_embedding = embed_text(query)

    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

In [51]:
query = "Describe the content of this document."

retrieved_docs = retrieve_multimodal(query, k=2)

message = create_multimodal_message(
    query,
    retrieved_docs
)

print("Multimodal message created successfully!")
print("Number of content blocks:", len(message.content))

Multimodal message created successfully!
Number of content blocks: 5


In [52]:

def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for Multimodal RAG using Gemini."""

    # 1. Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)

    # 2. Create multimodal message
    message = create_multimodal_message(
        query,
        context_docs
    )

    # 3. Get response from Gemini
    response = llm.invoke([message])

    # 4. Print retrieved context information
    print(f"\nRetrieved {len(context_docs)} documents:")

    for doc in context_docs:

        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")

        if doc_type == "text":

            preview = (
                doc.page_content[:100] + "..."
                if len(doc.page_content) > 100
                else doc.page_content
            )

            print(f"  - Text from page {page + 1}: {preview}")

        else:
            print(f"  - Image from page {page + 1}")

    print("\nGemini Response:\n")

    return response.content


In [53]:

if __name__ == "__main__":

    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]

    for query in queries:

        print(f"\nQuery: {query}")
        print("-" * 50)

        answer = multimodal_pdf_rag_pipeline(query)

        print(f"Answer: {answer}")
        print("=" * 70)



Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 1: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 1

Gemini Response:

Answer: [{'type': 'text', 'text': 'Based on the text and chart on page 1, the revenue grew steadily across Q1, Q2, and Q3, with the highest growth recorded in Q3:\n\n* **Q1** (blue bar)'}, {'type': 'text', 'text': ' showed a moderate increase in revenue due to new product lines.\n* **Q2** (green bar) outperformed Q1 due to marketing campaigns.\n* **Q3** (red bar) experienced exponential growth and the highest revenue due to global expansion.', 'extras': {'signature': 'Eo4BCosBAWkUfRO7mTRHVU4WRneX8xSgoSXmxXoCpZ5wSEEMT5mzH1DK7WJVymZIwQXk60q5e/5YSaWDZUL/RlueNB45fFtyl8ZZgDdOADFZtBi+ntG9N3QjS3r35W+C0R+Z2kZFrDHh4FcAgqvCY2h4anqcePkDL+zu3kgvulipY57tzqxEkYV5kcpVBsKXKg=='}}]

Query: Summarize t

In [54]:
query = "What is the main topic of this PDF?"

# Retrieve relevant text and images
retrieved_docs = retrieve_multimodal(query, k=5)

# Create a multimodal message
message = create_multimodal_message(query, retrieved_docs)

# Generate answer using Gemini
response = llm.invoke([message])

print(response.content)

[{'type': 'text', 'text': 'Based on the provided text, the main topic of the PDF is an **Annual Revenue Overview**, which summarizes the revenue trends across Q1, Q2, and Q3.', 'extras': {'signature': 'Eo4BCosBAWkUfRN5L0lC3KackhGCCiKUA0u3/TmmRHwJ1oDuaBugVOEZp/0m1Dcclg/VcyYhZsptnzDHk+vxFlkkfFclJXcMh32MBBr1O/EHns5r274/lgTwSnuJmNL8W4l1b1GKHBIdDIrOLvP3JxdOmvUE/XBgiuFliE9hLjq+wO/ZLFY3vInu5v/YbJt57A=='}}]


In [55]:
query = "Describe the chart or image in the PDF."

retrieved_docs = retrieve_multimodal(query, k=5)
message = create_multimodal_message(query, retrieved_docs)

response = llm.invoke([message])

print(response.content)

[{'type': 'text', 'text': 'Based on the provided document and image, the chart is a bar chart displaying revenue trends across three periods (Q1, Q2, and Q3). It consists of three vertical bars of increasing height from left to right:\n- A shorter **blue bar** representing Q1.\n- A medium **green bar** representing Q2.\n- The tallest **red bar** representing Q3, which shows the highest growth.', 'extras': {'signature': 'Eo4BCosBAWkUfROs2XvmsgcGjUN6prj+sYNgur5d2fxrBeko842IuBEDdaIAtd6t/wfQqwHNrjEGMuML7aMBxmoeQ5NlanQjahCF5k/H+MNGnTYRMCGFEalcC/2UDY8XTsg8+jytxoj2t4OfBEt7jd0ADsbaDXdqWOunhSNWN5DKq99NB49xcuJYPsaFi4HbNw=='}}]
